In [12]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..' ,'..')))
from Data.class_dataset import MRIDataset
from Model.model import build_vit3d

# --- Cargar datos de test ---

df=pd.read_csv("ext_test_data.csv")

test_imgs = df["Path"].tolist()
test_ages = df["Age"].tolist()
test_dataset = MRIDataset(test_imgs, test_ages)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# --- Cargar modelo ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_vit3d()
state_dict = torch.load("../../Training/Trained_models/model_8.pth", map_location=device)
# Si las claves tienen 'module.' al inicio, elimínalo
if any(k.startswith('module.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace('module.', '', 1)
        new_state_dict[new_key] = v
    state_dict = new_state_dict
model.load_state_dict(state_dict)
model.to(device)
model.eval()



ViT3D(
  (patch_to_embedding): Linear(in_features=4096, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=512, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=512, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=512, out_features=1024, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=0.1, i

In [13]:
# --- Evaluar ---
all_preds = []
all_ages = []
with torch.no_grad():
    for imgs, ages in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        all_preds.append(preds.cpu())
        all_ages.append(ages.unsqueeze(1).cpu())
all_preds = torch.cat(all_preds).numpy().flatten()
all_ages = torch.cat(all_ages).numpy().flatten()
all_paths = df["Path"].tolist()

mae = mean_absolute_error(all_ages, all_preds)
r2 = r2_score(all_ages, all_preds)
print(f"Test MAE: {mae:.2f}")
print(f"Test R2: {r2:.2f}")

Test MAE: 6.39
Test R2: 0.80


In [14]:
#crear un df con las edades reales, predichas y el ID
results_df = pd.DataFrame({
    "ID": all_paths,
    "Age": all_ages,
    "Prediction": all_preds,
    "Error": all_preds - all_ages, 
    'Absolute Error': np.abs(all_preds - all_ages)
})
results_df['ID']=results_df['ID'].replace('/data/lautaro/quasiraw/', '', regex=True)
results_df['ID']=results_df['ID'].replace('.nii.gz', '', regex=True)
results_df.sort_values(by='Absolute Error', ascending=True, inplace=True)
results_df.to_csv("test_predictions_6.csv", index=False)

In [15]:
results_df

,ID,Age,Prediction,Error,Absolute Error
1201,/data/lautaro/quasiraw_ext/116_S_0382_I1037254,88.0,88.000877,0.000877,0.000877
1131,/data/lautaro/quasiraw_ext/099_S_6038_I905866,78.0,77.994232,-0.005768,0.005768
93,/data/lautaro/quasiraw_ext/CP0112,46.0,45.992409,-0.007591,0.007591
725,/data/lautaro/quasiraw_ext/020_S_6185_I1117156,84.0,83.987801,-0.012199,0.012199
1160,/data/lautaro/quasiraw_ext/100_S_6493_I1236425,81.0,80.982666,-0.017334,0.017334
...,...,...,...,...,...
463,/data/lautaro/quasiraw_ext/RRIB_sub-141,25.0,52.992783,27.992783,27.992783
477,/data/lautaro/quasiraw_ext/RRIB_sub-155,43.0,14.047462,-28.952538,28.952538
231,/data/lautaro/quasiraw_ext/JUK_sub-40,25.0,56.884201,31.884201,31.884201
757,/data/lautaro/quasiraw_ext/021_S_6987_I1475785,66.0,33.710743,-32.289257,32.289257
